# Greenwashing Quantification Engine — analysis notebook

**MSc Financial Technology dissertation · University of Birmingham**

This notebook reproduces every number reported in Chapter 4 of the dissertation,
starting from the published data files. It runs top to bottom with no setup: the
first cell downloads the data from a GitHub repository into this
notebook's own runtime.

**To run it:** File → Save a copy in Drive (or Runtime → Run all). You do not
need a Google account that has been given access to anything, and you do not
need to mount your own Drive. The data folder is shared read-only with anyone
holding the link.

**Runtime:** about three minutes on a free CPU instance. No GPU needed.

| Section | Produces |
|---|---|
| 1 | Setup and data download |
| 2 | Descriptive statistics — Table 4.1 |
| 3 | GDS ranking and extreme cases — Table 4.2, Appendix A |
| 4 | Robustness: weighting and drivers — Table 4.3 |
| 5 | Event study — Tables 4.4 and 4.5 |
| 6 | Regression with controls — Table 4.6 |
| 7 | Sector-adjusted GDS — Table 4.7 |
| 8 | Thematic analysis and FDR correction — Tables 4.8 and 4.9 |
| 9 | Correlation matrix — Appendix D |

The language-scoring stage (FinBERT over the BRSR corpus) is in a separate
notebook, `GQE_LPS_scoring.ipynb`, because it needs the PDF corpus rather than
the CSV files. Its output, `LPS_scores.csv`, is one of the inputs here.

## 1. Setup

The folder link below is a read-only share. `gdown` pulls the files into this
runtime, so nothing is written to anyone's Drive and no sign-in is required.

In [ ]:
# ── SETUP: pull code + data from GitHub (no Drive, no sign-in) ──────────────
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/xbhxnxv/GQE-dissertation.git"
REPO_DIR = "/content/GQE-dissertation"
DATA_DIR = REPO_DIR + "/data"

if not os.path.exists(DATA_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

DATA = Path(DATA_DIR)
files = sorted(p.name for p in DATA.iterdir() if p.is_file())
print(f"\n{len(files)} files in {DATA_DIR}:")
for f in files:
    print("   ", f)


In [ ]:
# ── LIBRARIES ────────────────────────────────────────────────────────────────
for pkg, imp in [("pandas", "pandas"), ("numpy", "numpy"), ("scipy", "scipy"),
                 ("statsmodels", "statsmodels"), ("scikit-learn", "sklearn"),
                 ("xgboost", "xgboost"), ("shap", "shap"), ("openpyxl", "openpyxl")]:
    ensure(pkg, imp)

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

def load(name):
    """Read a data file from the shared folder by filename."""
    p = DATA / name
    if not p.exists():
        raise FileNotFoundError(f"{name} is not in the shared folder. Files present: "
                                f"{sorted(q.name for q in DATA.iterdir())}")
    return pd.read_excel(p) if p.suffix == ".xlsx" else pd.read_csv(p)

gds       = load("GDS_FINAL.csv")          # canonical score, 193 company-years
scores    = load("GDS_scores.csv")         # raw and z-scored OPS components
panel     = load("_analysis_panel.csv")    # sector, size, leverage
cars      = load("event_study_CARs.csv")   # abnormal returns by window
topics    = load("topic_assignments.csv")  # LDA topic shares per filing

print(f"GDS      {len(gds):>6} company-years, {gds['company'].nunique()} companies")
print(f"CARs     {len(cars):>6} event windows")
print(f"topics   {len(topics):>6} filings")

### Company-name crosswalk

The operational tracker abbreviates company names ("BPCL", "TCS") where the
disclosure files use full legal names ("Bharat Petroleum", "Tata Consultancy
Services"). Joining the two naively drops about 40% of the sample without
warning. `GDS_scores.csv` carries both spellings, so it is reused as the
crosswalk rather than matching the names a second time.

In [ ]:
XWALK = dict(zip(scores["company"], scores["company_ops"]))

def to_tracker_names(df, col="company"):
    out = df.copy()
    out[col] = out[col].map(lambda c: XWALK.get(c, c))
    return out

topics = to_tracker_names(topics)
missing = sorted(set(topics["company"]) - set(gds["company"]))
print("companies with topic shares but no GDS row:", missing or "none")

## 2. Descriptive statistics — Table 4.1

LPS and OPS are reported raw. Only GDS is standardised, and it is standardised
within each fiscal year, which is why GDS is the only row with a mean of zero.
A zero mean is a property of the construction, not a finding about the firms.

In [ ]:
rows = []
for col, label, scale in [("LPS_finbert", "LPS (FinBERT)", "raw"),
                          ("OPS", "OPS", "raw"),
                          ("GDS", "GDS", "standardised")]:
    for year, g in gds.groupby("year"):
        x = g[col].dropna()
        rows.append({"variable": label, "scale": scale, "year": year, "n": len(x),
                     "mean": x.mean(), "sd": x.std(ddof=1), "min": x.min(),
                     "p25": x.quantile(.25), "median": x.median(),
                     "p75": x.quantile(.75), "max": x.max()})

desc = pd.DataFrame(rows).round(4)
display(desc)

paired = gds.groupby("company")["year"].nunique().eq(2).sum()
print(f"companies with both years (paired sample): {paired}")

## 3. The ranking — Table 4.2 and Appendix A

GDS = z(LPS) − z(OPS). A high score can come from positive language, weak
operational performance, or both, and the component columns show which.

In [ ]:
ranked = (gds.merge(panel[["company", "year", "sector"]], on=["company", "year"], how="left")
             .sort_values("GDS", ascending=False)
             .reset_index(drop=True))
ranked.index += 1

cols = ["company", "year", "sector", "LPS_finbert", "OPS", "zLPS", "zOPS", "GDS"]
print("TEN LARGEST GAPS")
display(ranked[cols].head(10).round(4))
print("TEN SMALLEST GAPS")
display(ranked[cols].tail(10).round(4))

In [ ]:
# Full 193-row ranking (Appendix A)
display(ranked[cols].round(4))

## 4. Robustness — Table 4.3

**Weighting.** OPS gives every available component equal weight, which is a
choice. The score is rebuilt with weights taken from the first principal
component and the two versions are correlated. Missing values are median-filled
for the PCA fit only; the equal-weighted score is untouched.

**Drivers.** Three models are fitted on the same features to avoid resting the
answer on one model class, and SHAP values attribute each prediction back to its
inputs. This is diagnostic, not predictive. Log emissions is included on purpose
even though emissions feed OPS: the point is to measure how much of the score is
emissions.

In [ ]:
from sklearn.decomposition import PCA

ZCOLS = [c for c in ["_z_scope1", "_z_scope12", "_z_energy", "_z_renew_energy",
                     "_z_renew_pct", "_z_women_dir", "_z_women_emp", "_z_controversies"]
         if c in scores.columns and scores[c].notna().sum() > 50]

filled = scores[ZCOLS].apply(lambda c: c.fillna(c.median()))
pca = PCA(n_components=1).fit(filled)
w = pca.components_[0]
if np.corrcoef(filled @ w, scores["OPS"])[0, 1] < 0:
    w = -w                                     # PC sign is arbitrary
ops_pca = filled @ w / np.abs(w).sum()

r, p = stats.pearsonr(scores["OPS"], ops_pca)
print(f"PC1 explains {pca.explained_variance_ratio_[0]:.1%} of component variance")
print(f"equal-weighted vs PCA-weighted OPS:  r = {r:.3f}  (p = {p:.2e}, n = {len(scores)})")
display(pd.Series(w / np.abs(w).sum(), index=ZCOLS, name="PC1 weight")
          .sort_values(key=abs, ascending=False).round(3))

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb, shap

TOPIC_NAMES = {"topic_0_share": "Workforce",        "topic_1_share": "Customers/data",
               "topic_2_share": "Waste/packaging",  "topic_3_share": "Energy/water/emis",
               "topic_4_share": "Public policy",    "topic_5_share": "Health&safety",
               "topic_6_share": "Ethics/anti-corr", "topic_7_share": "Stakeholder/comm"}

em = (scores[["company_ops", "year", "scope12"]]
        .rename(columns={"company_ops": "company"}))

design = (gds.merge(em, on=["company", "year"], how="left")
             .merge(panel[["company", "year", "log_market_cap", "leverage"]],
                    on=["company", "year"], how="left")
             .merge(topics[["company", "year"] + list(TOPIC_NAMES)],
                    on=["company", "year"], how="left")
             .rename(columns={"log_market_cap": "Firm size (log mcap)",
                              "leverage": "Leverage", **TOPIC_NAMES}))
design["Emissions (log)"] = np.log1p(design["scope12"])

FEATS = ["Emissions (log)", "Firm size (log mcap)", "Leverage"] + list(TOPIC_NAMES.values())
d = design.dropna(subset=FEATS + ["GDS"])
X, y = d[FEATS], d["GDS"]
cv = KFold(5, shuffle=True, random_state=42)
print(f"design matrix: {X.shape[0]} company-years x {X.shape[1]} features\n")

lasso = LassoCV(cv=cv, random_state=42, max_iter=20000).fit(StandardScaler().fit_transform(X), y)
rf    = RandomForestRegressor(n_estimators=500, min_samples_leaf=3, random_state=42, n_jobs=-1)
gbm   = xgb.XGBRegressor(n_estimators=400, max_depth=3, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8, random_state=42)
print(f"LASSO          R2(cv) = {cross_val_score(lasso, StandardScaler().fit_transform(X), y, cv=cv).mean():.3f}")
print(f"Random Forest  R2(cv) = {cross_val_score(rf, X, y, cv=cv).mean():.3f}")
print(f"XGBoost        R2(cv) = {cross_val_score(gbm, X, y, cv=cv).mean():.3f}\n")

gbm.fit(X, y)
sv = shap.TreeExplainer(gbm).shap_values(X)
imp = (pd.DataFrame({"feature": FEATS, "mean_abs_shap": np.abs(sv).mean(axis=0)})
         .sort_values("mean_abs_shap", ascending=False))
display(imp.round(3))

## 5. Event study — Tables 4.4 and 4.5

Abnormal returns come from a market model estimated against the NIFTY 500 over
250 trading days ending eleven days before each disclosure. The NIFTY 500 rather
than the NIFTY 100 because every sample firm is a NIFTY 100 constituent, so
benchmarking against that index would place each firm inside its own market
proxy.

The cell below recomputes the CARs from raw prices. Re-running it is the point:
`event_study_CARs.csv` in the shared folder should come back byte for byte.

In [ ]:
EST_START, EST_END, MIN_OBS = -260, -11, 100
WINDOWS = {"AR_0": (0, 0), "CAR_m1_p1": (-1, 1), "CAR_0_p1": (0, 1),
           "CAR_0_p5": (0, 5), "CAR_m5_p5": (-5, 5), "CAR_m5_p20": (-5, 20)}

prices = pd.read_csv(DATA / "stock_prices.csv", parse_dates=["date"])
bench  = (pd.read_csv(DATA / "nifty500_benchmark.csv", parse_dates=["date"])
            [["date", "daily_return"]].rename(columns={"daily_return": "mkt_return"}))
events = (pd.read_csv(DATA / "disclosure_dates.csv", parse_dates=["event_date"])
            .merge(gds[["company", "year", "GDS"]], on=["company", "year"], how="inner")
            .dropna(subset=["event_date"]))

rows, skipped = [], []
for company, ev_firm in events.groupby("company"):
    pan = prices[prices["company"] == company].sort_values("date").reset_index(drop=True)
    if pan.empty:
        skipped.append((company, "no price series")); continue
    pan = pan.merge(bench, on="date", how="left")

    for _, e in ev_firm.iterrows():
        after = pan.index[pan["date"] >= e["event_date"]]
        if len(after) == 0:
            skipped.append((company, f"{e['year']}: event after price history")); continue
        t0 = int(after[0])

        est = pan.iloc[max(0, t0 + EST_START):t0 + EST_END + 1].dropna(
            subset=["daily_return", "mkt_return"])
        if len(est) < MIN_OBS:
            skipped.append((company, f"{e['year']}: only {len(est)} estimation days")); continue

        beta, alpha = np.polyfit(est["mkt_return"], est["daily_return"], 1)
        ab = pan["daily_return"] - (alpha + beta * pan["mkt_return"])

        rec = {"company": company, "year": e["year"],
               "event_date": e["event_date"].date(), "GDS": e["GDS"],
               "alpha": alpha, "beta": beta, "n_est": len(est)}
        for name, (a, b) in WINDOWS.items():
            lo, hi = t0 + a, t0 + b
            rec[name] = ab.iloc[lo:hi + 1].sum() if (lo >= 0 and hi < len(pan)) else np.nan
        rows.append(rec)

recomputed = pd.DataFrame(rows)
print(f"usable event windows: {len(recomputed)}")
for co, why in skipped:
    print(f"   skipped  {co:<32} {why}")

In [ ]:
LABEL = {"CAR_m1_p1": "(-1,+1)", "CAR_0_p5": "(0,+5)",
         "CAR_m5_p5": "(-5,+5)", "CAR_m5_p20": "(-5,+20)"}

print("Table 4.4  Mean CAR, market model vs NIFTY 500\n")
print(f"{'window':<10}{'n':>5}{'mean CAR':>11}{'95% CI':>22}{'t':>8}{'p':>8}{'d':>8}")
for w in LABEL:
    x = cars[w].dropna()
    m, sd, n = x.mean(), x.std(ddof=1), len(x)
    se = sd / np.sqrt(n)
    t, p = stats.ttest_1samp(x, 0.0)
    ci = f"[{(m-1.96*se)*100:+.2f}%, {(m+1.96*se)*100:+.2f}%]"
    print(f"{LABEL[w]:<10}{n:>5}{m*100:>10.2f}%{ci:>22}{t:>8.2f}{p:>8.3f}{m/sd:>8.2f}")

cut = cars["GDS"].median()
print("\n\nTable 4.5  High-GDS minus low-GDS portfolio CAR (median split)\n")
print(f"{'window':<10}{'difference':>12}{'95% CI':>24}{'Welch t':>10}{'p':>8}")
for w in ["CAR_m1_p1", "CAR_m5_p5", "CAR_m5_p20"]:
    hi = cars.loc[cars["GDS"] > cut, w].dropna()
    lo = cars.loc[cars["GDS"] <= cut, w].dropna()
    diff = hi.mean() - lo.mean()
    se = np.sqrt(hi.var(ddof=1) / len(hi) + lo.var(ddof=1) / len(lo))
    t, p = stats.ttest_ind(hi, lo, equal_var=False)
    ci = f"[{(diff-1.96*se)*100:+.2f}%, {(diff+1.96*se)*100:+.2f}%]"
    print(f"{LABEL[w]:<10}{diff*100:>11.2f}%{ci:>24}{t:>10.2f}{p:>8.3f}")

print("\n\nContinuous association: Pearson r between GDS and CAR\n")
for w in LABEL:
    d2 = cars[[w, "GDS"]].dropna()
    r, p = stats.pearsonr(d2["GDS"], d2[w])
    print(f"  {LABEL[w]:<10} n={len(d2):>4}   r = {r:+.3f}   p = {p:.3f}")

Every point estimate is negative and none reaches conventional significance.
With around 190 observations over two reporting years the design can only detect
a medium or larger effect, so this is an underpowered result rather than a null.

The event windows also overlap heavily, which the next cell quantifies. Overlap
means abnormal returns are cross-sectionally correlated and the standard errors
above are optimistic — which strengthens the reading rather than weakening it,
since results that already miss significance under optimistic errors would miss
more clearly under correct ones.

In [ ]:
dates = pd.to_datetime(cars["event_date"])
counts = dates.value_counts()
arr = dates.values
overlap = sum(bool(((np.abs((arr - arr[i]).astype("timedelta64[D]").astype(int)) <= 25)
                    & (np.arange(len(arr)) != i)).any()) for i in range(len(arr)))

print(f"event windows                                  {len(cars)}")
print(f"distinct disclosure dates                      {dates.nunique()}")
print(f"firm-years sharing an exact date with another  {int(counts[counts > 1].sum())}")
print(f"firm-years whose (-5,+20) window overlaps      {overlap}")
print(f"date range                                     {dates.min().date()} to {dates.max().date()}")

## 6. Regression with controls — Table 4.6

CAR regressed on GDS with firm size, leverage and sector dummies, estimated by
pooled OLS with HC3 robust standard errors. Pooled rather than firm fixed
effects because two years per firm leaves too little within-firm variation for
fixed effects to use.

In [ ]:
print("CAR = b0 + b1*GDS + b2*log(mcap) + b3*leverage + sector dummies + e   (HC3)\n")
print(f"{'window':<10}{'n':>5}{'GDS coef':>12}{'robust SE':>12}{'t':>8}{'p':>8}{'R2':>8}{'adj R2':>9}")
for w in LABEL:
    d3 = cars[[w, "GDS", "log_mcap", "leverage", "sector"]].dropna()
    X3 = pd.get_dummies(d3[["sector"]], drop_first=True).astype(float)
    X3[["GDS", "log_mcap", "leverage"]] = d3[["GDS", "log_mcap", "leverage"]].values
    fit = sm.OLS(d3[w].values, sm.add_constant(X3)).fit(cov_type="HC3")
    print(f"{LABEL[w]:<10}{int(fit.nobs):>5}{fit.params['GDS']:>12.5f}{fit.bse['GDS']:>12.5f}"
          f"{fit.tvalues['GDS']:>8.2f}{fit.pvalues['GDS']:>8.3f}"
          f"{fit.rsquared:>8.3f}{fit.rsquared_adj:>9.3f}")

## 7. Sector-adjusted GDS — Table 4.7

The top of the ranking is heavy industry and OPS is emissions-driven, so the
obvious challenge is whether GDS measures greenwashing or simply measures being
a steel mill. Subtracting the sector mean removes every cross-sector difference.
Whatever survives is a within-sector effect: a firm judged only against its
own peers.

In [ ]:
sec = (gds.merge(panel[["company", "year", "sector"]], on=["company", "year"], how="left")
          .merge(em, on=["company", "year"], how="left"))
sec["GDS_sector_adj"] = sec["GDS"] - sec.groupby("sector")["GDS"].transform("mean")
sec["log_emissions"]  = np.log1p(sec["scope12"])

r_raw, p_raw = stats.pearsonr(sec["log_emissions"], sec["GDS"])
r_adj, p_adj = stats.pearsonr(sec["log_emissions"], sec["GDS_sector_adj"])
print(f"n = {len(sec)},  sectors = {sec['sector'].nunique()}\n")
print(f"log(Scope 1+2) vs raw GDS             r = {r_raw:+.3f}   p = {p_raw:.2e}")
print(f"log(Scope 1+2) vs sector-adjusted GDS r = {r_adj:+.3f}   p = {p_adj:.4f}")

adj = sec[["company", "year", "GDS_sector_adj"]]
c7 = cars.merge(adj, on=["company", "year"], how="left")
d7 = c7[["CAR_m5_p20", "GDS_sector_adj", "log_mcap", "leverage"]].dropna()
fit7 = sm.OLS(d7["CAR_m5_p20"],
              sm.add_constant(d7[["GDS_sector_adj", "log_mcap", "leverage"]])).fit(cov_type="HC3")
print(f"\nCAR(-5,+20) on sector-adjusted GDS:  b = {fit7.params['GDS_sector_adj']:+.5f}  "
      f"t = {fit7.tvalues['GDS_sector_adj']:+.2f}  p = {fit7.pvalues['GDS_sector_adj']:.3f}")

print("\nSector means, descending:")
display(sec.groupby("sector")["GDS"].agg(["count", "mean", "std"])
           .sort_values("mean", ascending=False).round(3))

## 8. Thematic analysis — Tables 4.8 and 4.9

Sixteen tests are run: eight LDA themes against GDS, and the same eight against
the standardised operational component. Reporting only the significant ones out
of sixteen would manufacture findings, so a Benjamini-Hochberg false discovery
rate correction is applied across the whole family, with Bonferroni alongside as
the conservative bound.

In [ ]:
def benjamini_hochberg(pvals):
    """Step-up FDR adjustment with monotonicity enforced."""
    p = np.asarray(pvals, float)
    n = p.size
    order = np.argsort(p)
    adj = p[order] * n / (np.arange(n) + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    out = np.empty(n)
    out[order] = np.minimum(adj, 1.0)
    return out

LABELS = {0: "Workforce/wages/rights",     1: "Customers/product/data",
          2: "Waste/packaging/circularity", 3: "Energy/water/emissions",
          4: "Public policy advocacy",      5: "Health & safety",
          6: "Ethics/anti-corruption",      7: "Stakeholder/community"}

# GDS_FINAL already carries zLPS and zOPS, so no second merge is needed
tt = topics.merge(gds, on=["company", "year"])

rows = []
for k, label in LABELS.items():
    col = f"topic_{k}_share"
    rg, pg = stats.pearsonr(tt[col], tt["GDS"])
    ro, po = stats.pearsonr(tt[col], tt["zOPS"])
    rows.append({"theme": label, "r_GDS": rg, "p_GDS": pg, "r_zOPS": ro, "p_zOPS": po})
tc = pd.DataFrame(rows)

names = [f"{r.theme} x GDS" for r in tc.itertuples()] + [f"{r.theme} x zOPS" for r in tc.itertuples()]
rs    = list(tc["r_GDS"]) + list(tc["r_zOPS"])
praw  = np.array(list(tc["p_GDS"]) + list(tc["p_zOPS"]))
pfdr, pbon = benjamini_hochberg(praw), np.minimum(praw * len(praw), 1.0)

fam = (pd.DataFrame({"test": names, "r": np.round(rs, 3), "p_raw": praw,
                     "p_FDR": pfdr, "p_Bonferroni": pbon,
                     "survives_FDR": np.where(pfdr < 0.05, "yes", "no")})
         .sort_values("p_raw").reset_index(drop=True))
display(fam.round(4))
print(f"survive FDR (q<0.05): {(pfdr < 0.05).sum()} / {len(praw)}")
print(f"survive Bonferroni:   {(pbon < 0.05).sum()} / {len(praw)}")

In [ ]:
# Table 4.9 — which theme associations survive sector adjustment
ts = topics.merge(sec[["company", "year", "GDS", "GDS_sector_adj"]], on=["company", "year"])
rows = []
for k, label in LABELS.items():
    col = f"topic_{k}_share"
    r1, p1 = stats.pearsonr(ts[col], ts["GDS"])
    r2, p2 = stats.pearsonr(ts[col], ts["GDS_sector_adj"])
    rows.append({"theme": label, "r_raw": round(r1, 3), "p_raw": round(p1, 4),
                 "r_sector_adj": round(r2, 3), "p_sector_adj": round(p2, 4),
                 "survives": "yes" if p2 < 0.05 else "no"})
display(pd.DataFrame(rows))

Four themes hold within sector: energy and emissions, ethics, workforce, and
stakeholder emphasis. Waste and packaging and public policy advocacy do not,
which means those were cross-sector patterns rather than firm-level deflection.
Reporting both halves is the honest reading of the measure.

## 9. Correlation matrix — Appendix D

Pearson correlations, computed pairwise, so the number of observations varies by
cell: 193 for the complete components, fewer for the renewable and gender
variables, and 191 or 183 for the CAR columns.

In [ ]:
ops_raw = (scores[["company_ops", "year", "scope12", "energy", "renew_pct",
                   "women_dir", "women_emp", "controversies"]]
             .rename(columns={"company_ops": "company"}))

cm = (gds.merge(panel[["company", "year", "log_market_cap", "leverage"]], on=["company", "year"])
         .merge(ops_raw, on=["company", "year"])
         .merge(cars[["company", "year", "CAR_m1_p1", "CAR_m5_p20"]], on=["company", "year"], how="left"))
cm["log_emissions"] = np.log1p(cm["scope12"])
cm["log_energy"]    = np.log1p(cm["energy"])

V = ["LPS_finbert", "OPS", "GDS", "log_emissions", "log_energy", "renew_pct",
     "women_dir", "women_emp", "controversies", "log_market_cap", "leverage",
     "CAR_m1_p1", "CAR_m5_p20"]
display(cm[V].corr().round(3))
print("\nnon-missing observations per variable:")
print(cm[V].notna().sum().to_string())

---

### Notes on reproduction

Everything above runs from published CSV files. The one stage not reproduced
here is the language scoring itself, which needs the BRSR PDF corpus and a
FinBERT model download; that is in `GQE_LPS_scoring.ipynb`.

Two figures reported in the dissertation do not reproduce exactly and are
flagged rather than hidden:

- The SHAP importance for emissions is 0.73 in the reported table and around
  0.69 here. The difference comes from library versions of `xgboost` and `shap`,
  not from the data. The ordering and the conclusion are unchanged.
- The PCA weighting correlation is reported as 0.87 on 86 complete cases. The
  full-sample median-filled version above gives 0.865.

Data files, methodology and full results are described in the dissertation.
Questions to the author.